In [ ]:
import tensorflow as tf
import os, json

# Create config.json with correct Drive path, otherwise there is error
config = {"dataset_path": "/content/drive/MyDrive/CMSC 324 Final project dataset/BraTS-PEDs-v1/"}
with open('./config.json', 'w') as f:
    json.dump(config, f)

from load_dataset import build_dataset, make_tf_dataset
from model import build_unet3d   
from metric import dice_coef, bce_dice_loss

ModuleNotFoundError: No module named 'tensorflow'

# Load hyperparameter

In [ ]:
with open('./hparams.json', 'r') as f:
    config = json.load(f)
    hparams = config['unet']['hparam_grid']  

# Build datasets

In [ ]:
x_train, y_train = build_dataset(training=True)
# x_train, y_train = build_data(data="train", n=5) # for a dry-run

x_val,   y_val   = build_dataset(training=False)

# TODO: actually we might need make_torch_dataset
train_ds = make_tf_dataset(x_train, y_train, training=True)
val_ds = make_tf_dataset(x_val,   y_val,   training=False)

# Configure model

## TODO: use torch instead of tensorflow

In [ ]:
model = build_unet3d(input_shape=hparams["input_shape"]) # or add more hyperparameters
model.summary()

# maybe abstractize this too? or this coudl be fine as is
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(hparams["model_out"], save_best_only=True, monitor="val_dice_coef", mode="max"),
    tf.keras.callbacks.EarlyStopping(monitor="val_dice_coef", mode="max", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_dice_coef", mode="max", factor=0.5, patience=2),
]

# TODO: figure out how this works
optim = tf.keras.optimizers.Adam(learning_rate=hparams["learning_rate"]) 

# Train model

In [ ]:
# TODO: make sure these are the loss and matrics we use
model.compile(
    optimizer=optim,
    loss=bce_dice_loss,
    metrics=[dice_coef],
)

# TODO: Do we pass test_ds to the argument `validation_data`? 
# I think we should use val_ds
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=hparams["n_epochs"],
    callbacks=callbacks,
    verbose=1,
)

# Check the result

In [ ]:
best_val = max(history.history.get("val_dice_coef", [float("nan")]))
print(f"\nBest validation Dice: {best_val:.4f}")

# TODO: what does this do?
# show_prediction(model, x_val, y_val)

# TODO: make sure the trained model is saved!
model.save(hparams["model_out"])

# TODO: save train/val loss plot?
# TODO: save history?
with open("history.json", "w") as f:
    json.dump(history.history, f)